# Team Ensemble — Blending Individual Submissions

Combine each team member's best individual submission into a single prediction.

## Why ensembling works

Each member trained a different model (different features, different algorithm, different HPO).  
Even if one model is weaker individually, if its **errors are different** from others,  
blending reduces overall error — the same reason stacking XGB+LGBM+ET beats any single model.

## Three blend strategies (all tried below)

| Strategy | How | Best when |
|---|---|---|
| Equal weight | Simple average | Models are similarly strong |
| Weighted by RMSE | Better score = more weight (1/RMSE²) | Models differ significantly in strength |
| Rank average | Average prediction ranks | Models predict at different scales |

**Rule of thumb:** Submit the equal-weight blend first. If it beats everyone's individual best, you're done.

---
## 1. Load Team Submissions

Update `TEAM_FILES` with each member's **best** submission and their Kaggle RMSE.

In [ ]:
import pandas as pd
import numpy as np

SUBMISSION_DIR = '../outputs/submissions/Team Submission/'

# Format: (filepath, kaggle_rmse, member_name)
# MengHai upgraded to v20 (all Optuna params — LGBM+ET+CatBoost, OOF $21,465)
# Lai excluded — weighted blend failed ($21,660.70); rank avg not submitted
TEAM_FILES = [
    (SUBMISSION_DIR + '20260428_fulldataV1_lanson.csv',  21035, 'Lanson'),
    (SUBMISSION_DIR + '20260429_21827_MengHai.csv',      21827, 'MengHai'),  # v20 Kaggle $21,827
    (SUBMISSION_DIR + '20260427_22094_likhong.csv',      22094, 'Likhong'),
]

# Load all submissions
dfs = {}
for fpath, rmse_score, name in TEAM_FILES:
    df = pd.read_csv(fpath)
    dfs[name] = df.set_index('Id')['Predicted']
    print(f'{name:12s} | RMSE: ${rmse_score:>7,.0f} | Rows: {len(df)} | Price range: ${df.Predicted.min():,.0f}–${df.Predicted.max():,.0f}')

# Align all on same index
sample = pd.read_csv('../outputs/submissions/sample_sub_reg.csv')
pred_df = pd.DataFrame({name: s for name, s in dfs.items()}).reindex(sample['Id'])
print(f'\nAlignment check — any nulls: {pred_df.isna().sum().sum()}')
print(pred_df.head(3))

---
## 2. Correlation Between Predictions

Lower correlation = more potential gain from blending.

In [9]:
print('Prediction correlation matrix:')
print(pred_df.corr().round(4))
print('\nNote: correlation < 0.99 = meaningful diversity for blending')

Prediction correlation matrix:
         Lanson  MengHai  Likhong
Lanson   1.0000   0.9988   0.9982
MengHai  0.9988   1.0000   0.9988
Likhong  0.9982   0.9988   1.0000

Note: correlation < 0.99 = meaningful diversity for blending


---
## 3. Strategy 1 — Equal Weight Blend

In [10]:
blend_equal = pred_df.mean(axis=1)
print(f'Equal-weight blend:')
print(f'  Mean price:  ${blend_equal.mean():,.0f}')
print(f'  Price range: ${blend_equal.min():,.0f} – ${blend_equal.max():,.0f}')

Equal-weight blend:
  Mean price:  $448,880
  Price range: $176,720 – $1,157,222


---
## 4. Strategy 2 — Weighted by Kaggle RMSE (1/RMSE²)

In [11]:
rmse_scores = {name: rmse for _, rmse, name in TEAM_FILES}
weights_raw = {name: 1 / (rmse**2) for name, rmse in rmse_scores.items()}
total_w     = sum(weights_raw.values())
weights     = {name: w / total_w for name, w in weights_raw.items()}

print('Normalised weights (1/RMSE²):')
for name, w in weights.items():
    print(f'  {name:12s}: {w:.4f}  (RMSE ${rmse_scores[name]:,.0f})')

blend_weighted = sum(pred_df[name] * w for name, w in weights.items())
print(f'\nWeighted blend:')
print(f'  Mean price:  ${blend_weighted.mean():,.0f}')
print(f'  Price range: ${blend_weighted.min():,.0f} – ${blend_weighted.max():,.0f}')

Normalised weights (1/RMSE²):
  Lanson      : 0.3500  (RMSE $21,035)
  MengHai     : 0.3327  (RMSE $21,578)
  Likhong     : 0.3173  (RMSE $22,094)

Weighted blend:
  Mean price:  $448,882
  Price range: $176,666 – $1,157,595


---
## 5. Strategy 3 — Rank Averaging

Average the rank (position) of each prediction rather than the raw value.  
More robust when members' models predict at very different price scales.

In [12]:
# Convert each member's predictions to ranks, then average ranks, then map back to prices
ranks_df = pred_df.rank()
avg_rank  = ranks_df.mean(axis=1).rank()  # final rank from averaged ranks

# Map averaged rank back to price scale using equal-weight blend as reference
ref_sorted  = blend_equal.sort_values().values
rank_sorted = avg_rank.sort_values().index
blend_rank  = pd.Series(ref_sorted, index=rank_sorted).reindex(sample['Id'])

print(f'Rank-average blend:')
print(f'  Mean price:  ${blend_rank.mean():,.0f}')
print(f'  Price range: ${blend_rank.min():,.0f} – ${blend_rank.max():,.0f}')

Rank-average blend:
  Mean price:  $448,880
  Price range: $176,720 – $1,157,222


---
## 6. Compare All Strategies

In [13]:
individual_best = min(rmse_scores.values())
best_member     = min(rmse_scores, key=rmse_scores.get)

print('=== Blend Strategy Comparison ===')
print(f'Individual best: ${individual_best:,.0f} ({best_member})')
print()
print('Individual predictions:')
for name, rmse in sorted(rmse_scores.items(), key=lambda x: x[1]):
    print(f'  {name:12s}: ${rmse:>7,.0f} (Kaggle)')
print()
print('Blends (submit to Kaggle to know true RMSE):')
print(f'  Equal weight : mean=${blend_equal.mean():,.0f}')
print(f'  RMSE weighted: mean=${blend_weighted.mean():,.0f}')
print(f'  Rank average : mean=${blend_rank.mean():,.0f}')
print()
print('Recommendation: start with equal-weight blend.')

=== Blend Strategy Comparison ===
Individual best: $21,035 (Lanson)

Individual predictions:
  Lanson      : $ 21,035 (Kaggle)
  MengHai     : $ 21,578 (Kaggle)
  Likhong     : $ 22,094 (Kaggle)

Blends (submit to Kaggle to know true RMSE):
  Equal weight : mean=$448,880
  RMSE weighted: mean=$448,882
  Rank average : mean=$448,880

Recommendation: start with equal-weight blend.


---
## 7. Save Ensemble Submissions

In [ ]:
date = '20260429'

def save_sub(predictions, filename, directory):
    sub = pd.DataFrame({'Id': sample['Id'], 'Predicted': predictions.reindex(sample['Id']).values})
    path = directory + filename
    sub.to_csv(path, index=False)
    print(f'Saved: {filename}  |  mean=${predictions.mean():,.0f}  |  rows={len(sub)}')

print('=== Team Submission folder ===')
save_sub(blend_equal,    f'{date}_3member_equal_v20.csv',    SUBMISSION_DIR)
save_sub(blend_weighted, f'{date}_3member_wt_v20.csv',       SUBMISSION_DIR)
save_sub(blend_rank,     f'{date}_3member_rank_v20.csv',     SUBMISSION_DIR)

print()
print('--- Submission priority ---')
print(f'1. {date}_3member_wt_v20.csv    — weighted (1/RMSE²); Lanson ~36%, MengHai ~34%, Likhong ~30%')
print(f'2. {date}_3member_equal_v20.csv — equal weight; simpler baseline')
print(f'3. {date}_3member_rank_v20.csv  — rank avg; robust to any scale differences')

---
## 8. Submitting Strategy

1. **Submit `sub_ensemble_equal.csv` first** — simplest, usually best
2. If it beats the individual best → great, you're done
3. If not, try `sub_ensemble_weighted.csv` (de-emphasises weaker members)
4. `sub_ensemble_rank.csv` is a tiebreaker if the others are similar

### As more members improve their models

Update `TEAM_FILES` in Cell 1 with each member's latest best file and re-run from Section 1.
The ensemble typically keeps improving as individual models improve.

### Key insight

If two members have very different approaches (e.g. one uses deep feature engineering,  
one uses raw features with aggressive HPO), their errors will be less correlated  
and the blend will improve more than if everyone used the same approach.